In [ ]:
import os
import sys
print(sys.version)
import pydicom
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers, models

In [ ]:
from  dicomutils import dicomutils
utils = dicomutils()

In [ ]:
target_size = (256, 256)


In [ ]:
from datautils import datautils
datautil = datautils()

X_train_rgb, y_train, X_val_rgb, y_val = datautil.load_data(target_size)

In [ ]:
# Modeli oluştur
input_shape = (target_size[0], target_size[1], 1)
model =utils.build_classification_model(input_shape, num_classes=3)

# Model özeti
model.summary()

# Modeli eğitme
history = model.fit(
    X_train_rgb[..., np.newaxis], y_train,
    validation_data=(X_val_rgb[..., np.newaxis], y_val),
    epochs=15,
    batch_size=32
)

In [ ]:
# Doğruluk ve kayıp grafikleri
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Eğitim Doğruluğu')
plt.plot(history.history['val_accuracy'], label='Doğrulama Doğruluğu')
plt.title('Model Doğruluğu')
plt.ylabel('Doğruluk')
plt.xlabel('Epok')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Eğitim Kaybı')
plt.plot(history.history['val_loss'], label='Doğrulama Kaybı')
plt.title('Model Kaybı')
plt.ylabel('Kayıp')
plt.xlabel('Epok')
plt.legend()
plt.show()

In [ ]:
# Modeli değerlendirme
test_loss, test_acc = model.evaluate(X_val_rgb[..., np.newaxis], y_val)
print(f"\nTest Doğruluğu: {test_acc:.4f}, Test Kaybı: {test_loss:.4f}")

# Karışıklık matrisi
from sklearn.metrics import confusion_matrix, classification_report
y_pred = model.predict(X_val_rgb[..., np.newaxis]).argmax(axis=1)
print("\nSınıflandırma Raporu:")
print(classification_report(y_val, y_pred, target_names=['Kanama', 'İskemi', 'Normal']))

In [ ]:
#modeli kaydetme
model.save('model.h5')

#modeli yükleme
# model = tf.keras.models.load_model('model.h5')


In [ ]:
# model = tf.keras.models.load_model('model.h5')

In [ ]:
# Test verisini yükleme
test_path = "YarısmaVeriSeti_1.Oturum/DICOM"
test_images = utils.load_dicom_data(test_path)

# Tahmin yapma
predictions = model.predict(test_images[..., np.newaxis])
pred_classes = predictions.argmax(axis=1)
class_names = ['Kanama', 'İskemi', 'Normal']

In [ ]:
#Doğruluk, Kesinlik, Duyarlılık ve F1 Skoru
# from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score 
# y_true = [0, 1, 2]  # Gerçek etiketler (örnek)
# y_pred = pred_classes  # Tahmin edilen etiketler
# accuracy = accuracy_score(y_true, y_pred)
# precision = precision_score(y_true, y_pred, average='weighted')
# recall = recall_score(y_true, y_pred, average='weighted')
# f1 = f1_score(y_true, y_pred, average='weighted')
# print(f"Doğruluk: {accuracy:.4f}")
# print(f"Kesinlik: {precision:.4f}")
# print(f"Duyarlılık: {recall:.4f}")
# print(f"F1 Skoru: {f1:.4f}")
